# Customer Churn Prediction — Stage 2: Feature Engineering + SQL Layer
## Day 3: RFM Scoring and New Features

Goal: Transform raw cleaned data into model-ready features.
We create RFM scores (a classic marketing framework), engineer domain-specific
features, and one-hot encode all categoricals.

**Why RFM?** Recency-Frequency-Monetary is a proven customer segmentation
framework used in marketing analytics. Adding it shows business domain knowledge
beyond just running ML models.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../src', exist_ok=True)

df = pd.read_csv('../data/telco_clean.csv')

print(f"Loaded telco_clean.csv — shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Loaded telco_clean.csv — shape: (7043, 22)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Churn_Binary']


## 1. Recency Score
Recency = how long the customer has been with us.
Lower tenure = newer = higher churn risk = LOWER recency score.
(Inverted so that higher RFM score always = better customer)

In [2]:
# Interview talking point: binning tenure into groups rather than using raw
# values makes the RFM score interpretable and robust to outliers.

def get_recency_score(tenure):
    if tenure <= 12:
        return 1
    elif tenure <= 24:
        return 2
    elif tenure <= 36:
        return 3
    elif tenure <= 48:
        return 4
    else:
        return 5

df['recency_score'] = df['tenure'].apply(get_recency_score)

print("Recency score distribution:")
print(df['recency_score'].value_counts().sort_index())
print("\nChurn rate per recency score:")
print(df.groupby('recency_score')['Churn_Binary'].mean().round(3))

Recency score distribution:
recency_score
1    2186
2    1024
3     832
4     762
5    2239
Name: count, dtype: int64

Churn rate per recency score:
recency_score
1    0.474
2    0.287
3    0.216
4    0.190
5    0.095
Name: Churn_Binary, dtype: float64


## 2. Frequency Score
Frequency = commitment level, measured by contract type.
Month-to-month = low commitment = highest churn risk.
Two year = high commitment = lowest churn risk.

In [3]:
frequency_map = {
    'Month-to-month': 1,
    'One year':       3,
    'Two year':       5
}

df['frequency_score'] = df['Contract'].map(frequency_map)

print("Frequency score distribution:")
print(df['frequency_score'].value_counts().sort_index())
print("\nChurn rate per frequency score:")
print(df.groupby('frequency_score')['Churn_Binary'].mean().round(3))

Frequency score distribution:
frequency_score
1    3875
3    1473
5    1695
Name: count, dtype: int64

Churn rate per frequency score:
frequency_score
1    0.427
3    0.113
5    0.028
Name: Churn_Binary, dtype: float64


## 3. Monetary Score
Monetary = how much revenue the customer generates.
We bin MonthlyCharges into 4 quartile groups so the score is relative
to the customer base, not absolute dollar amounts.

In [4]:
# pd.qcut creates equal-frequency bins (quartiles)
# Interview talking point: qcut vs cut — qcut ensures equal number of customers
# in each bin, making the score distribution balanced.

df['monetary_score'] = pd.qcut(
    df['MonthlyCharges'],
    q=4,
    labels=[1, 2, 3, 4],
    duplicates='drop'
).astype(int)

print("Monetary score distribution:")
print(df['monetary_score'].value_counts().sort_index())

# Show the actual charge ranges per quartile
print("\nMonthlyCharges ranges per monetary score:")
print(df.groupby('monetary_score')['MonthlyCharges'].agg(['min', 'max', 'mean']).round(2))

Monetary score distribution:
monetary_score
1    1762
2    1766
3    1757
4    1758
Name: count, dtype: int64

MonthlyCharges ranges per monetary score:
                  min     max    mean
monetary_score                       
1               18.25   35.50   22.23
2               35.55   70.35   55.73
3               70.40   89.85   80.33
4               89.90  118.75  100.90


## 4. RFM Composite Score
Sum of all three scores. Range: 3 (worst) to 14 (best).
Higher score = longer tenure + long-term contract + high spender = valuable, loyal customer.

In [5]:
df['rfm_score'] = df['recency_score'] + df['frequency_score'] + df['monetary_score']

print("RFM score distribution:")
print(df['rfm_score'].value_counts().sort_index())
print(f"\nMin RFM: {df['rfm_score'].min()} | Max RFM: {df['rfm_score'].max()}")
print("\nChurn rate per RFM score (higher score = lower churn ✓):")
print(df.groupby('rfm_score')['Churn_Binary'].mean().round(3))

RFM score distribution:
rfm_score
3     518
4     778
5     936
6     717
7     620
8     557
9     522
10    527
11    602
12    559
13    301
14    406
Name: count, dtype: int64

Min RFM: 3 | Max RFM: 14

Churn rate per RFM score (higher score = lower churn ✓):
rfm_score
3     0.297
4     0.407
5     0.500
6     0.417
7     0.308
8     0.223
9     0.186
10    0.163
11    0.047
12    0.125
13    0.020
14    0.071
Name: Churn_Binary, dtype: float64


## 5. Charges Per Service
Divides MonthlyCharges by number of active add-on services.
A customer paying $80/month with 1 service is overpaying vs one with 6 services.
This ratio captures perceived value — high ratio customers may feel ripped off and churn.

In [6]:
add_on_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies']

# Count how many add-ons each customer has (only count "Yes")
df['n_services'] = df[add_on_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)

# Avoid division by zero — customers with 0 add-ons get n_services=1 for this calc
df['charges_per_service'] = df['MonthlyCharges'] / df['n_services'].replace(0, 1)

print("charges_per_service stats:")
print(df['charges_per_service'].describe().round(2))
print(f"\nCustomers with 0 add-on services: {(df['n_services'] == 0).sum()}")

charges_per_service stats:
count    7043.00
mean       33.69
std        19.74
min        10.42
25%        19.95
50%        25.40
75%        44.22
max        87.05
Name: charges_per_service, dtype: float64

Customers with 0 add-on services: 2219


## 6. Binary Flag Features
Domain-knowledge features that directly encode high-risk customer profiles
identified during EDA.

In [7]:
# is_new_customer: tenure < 12 months = highest churn risk group from EDA
df['is_new_customer'] = (df['tenure'] < 12).astype(int)

# is_senior_no_support: seniors without tech support churn at very high rates
df['is_senior_no_support'] = (
    (df['SeniorCitizen'] == 'Yes') & (df['TechSupport'] == 'No')
).astype(int)

print("is_new_customer distribution:")
print(df['is_new_customer'].value_counts())
print(f"  Churn rate (new customers)    : {df[df['is_new_customer']==1]['Churn_Binary'].mean():.1%}")
print(f"  Churn rate (existing customers): {df[df['is_new_customer']==0]['Churn_Binary'].mean():.1%}")

print("\nis_senior_no_support distribution:")
print(df['is_senior_no_support'].value_counts())
print(f"  Churn rate (senior, no support): {df[df['is_senior_no_support']==1]['Churn_Binary'].mean():.1%}")
print(f"  Churn rate (others)            : {df[df['is_senior_no_support']==0]['Churn_Binary'].mean():.1%}")

is_new_customer distribution:
is_new_customer
0    4974
1    2069
Name: count, dtype: int64
  Churn rate (new customers)    : 48.3%
  Churn rate (existing customers): 17.5%

is_senior_no_support distribution:
is_senior_no_support
0    6213
1     830
Name: count, dtype: int64
  Churn rate (senior, no support): 50.6%
  Churn rate (others)            : 23.3%


## 7. One-Hot Encoding
Convert all remaining categorical string columns to numeric dummies.
We use drop_first=False to keep all categories — XGBoost handles multicollinearity
so we don't need to drop one column per group.

We save customerID, Churn, and Churn_Binary separately before encoding,
then add them back after — these are not model features.

In [8]:
# Columns to preserve (not encode)
preserve_cols = ['customerID', 'Churn', 'Churn_Binary']
preserve_df   = df[preserve_cols].copy()

# Columns to encode
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in preserve_cols]

print(f"Categorical columns to encode ({len(cat_cols)}):")
print(cat_cols)

# Drop preserve cols + Churn (string) before encoding
df_encode = df.drop(columns=preserve_cols).copy()

# One-hot encode
df_encoded = pd.get_dummies(df_encode, columns=cat_cols, drop_first=False)

# Add back the preserved columns
df_final = pd.concat([preserve_df.reset_index(drop=True),
                      df_encoded.reset_index(drop=True)], axis=1)

print(f"\nShape before encoding : {df.shape}")
print(f"Shape after encoding  : {df_final.shape}")
print(f"Total features        : {df_final.shape[1] - 3} (excluding customerID, Churn, Churn_Binary)")
print(f"Null values           : {df_final.isnull().sum().sum()}")

Categorical columns to encode (16):
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Shape before encoding : (7043, 30)
Shape after encoding  : (7043, 57)
Total features        : 54 (excluding customerID, Churn, Churn_Binary)
Null values           : 0


## 8. Save Feature Dataset

In [9]:
save_path = '../data/telco_features.csv'
df_final.to_csv(save_path, index=False)

print(f"✅ Saved → {save_path}")
print(f"   Shape : {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print(f"   Nulls : {df_final.isnull().sum().sum()}")
print(f"\nAll columns:")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:3}. {col}")

✅ Saved → ../data/telco_features.csv
   Shape : 7,043 rows × 57 columns
   Nulls : 0

All columns:
    1. customerID
    2. Churn
    3. Churn_Binary
    4. tenure
    5. MonthlyCharges
    6. TotalCharges
    7. recency_score
    8. frequency_score
    9. monetary_score
   10. rfm_score
   11. n_services
   12. charges_per_service
   13. is_new_customer
   14. is_senior_no_support
   15. gender_Female
   16. gender_Male
   17. SeniorCitizen_No
   18. SeniorCitizen_Yes
   19. Partner_No
   20. Partner_Yes
   21. Dependents_No
   22. Dependents_Yes
   23. PhoneService_No
   24. PhoneService_Yes
   25. MultipleLines_No
   26. MultipleLines_No phone service
   27. MultipleLines_Yes
   28. InternetService_DSL
   29. InternetService_Fiber optic
   30. InternetService_No
   31. OnlineSecurity_No
   32. OnlineSecurity_No internet service
   33. OnlineSecurity_Yes
   34. OnlineBackup_No
   35. OnlineBackup_No internet service
   36. OnlineBackup_Yes
   37. DeviceProtection_No
   38. DeviceProt

---
## Day 4: SQL Reporting Layer

Goal: Write all key insights into a SQLite database.
This demonstrates data engineering thinking — not just building an ML model,
but creating a queryable reporting layer that a BI team could use.

**Interview talking point:** Adding a SQL layer shows you think about
how data flows end-to-end in a real organisation, not just model building in isolation.

## 9. Create SQLite Database & Tables

In [10]:
db_path = '../outputs/churn_analysis.db'

# Remove old DB if exists (clean rebuild)
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)

# ── TABLE 1: customers ──────────────────────────────────────────────────
# Full feature set — used by the Streamlit dashboard
df_final.to_sql('customers', conn, if_exists='replace', index=False)
print(f"✅ Table 'customers' written — {len(df_final):,} rows")

# ── TABLE 2: churn_summary ──────────────────────────────────────────────
# Churn rate by Contract type and tenure group
df_summary = df[['Contract', 'tenure', 'MonthlyCharges', 'Churn_Binary']].copy()
df_summary['tenure_group'] = pd.cut(
    df_summary['tenure'],
    bins=[0, 12, 24, 36, 48, 72],
    labels=['0-12', '13-24', '25-36', '37-48', '49-72'],
    right=True
).astype(str)

churn_summary = (df_summary
    .groupby(['Contract', 'tenure_group'])
    .agg(
        churn_rate    = ('Churn_Binary', 'mean'),
        avg_monthly   = ('MonthlyCharges', 'mean'),
        customer_count= ('Churn_Binary', 'count')
    )
    .reset_index()
    .round(4))

churn_summary.to_sql('churn_summary', conn, if_exists='replace', index=False)
print(f"✅ Table 'churn_summary' written — {len(churn_summary)} rows")
print(churn_summary.head(8).to_string(index=False))

# ── TABLE 3: rfm_segments ───────────────────────────────────────────────
rfm_segments = (df_final
    .groupby('rfm_score')
    .agg(
        customer_count  = ('Churn_Binary', 'count'),
        avg_monthly     = ('MonthlyCharges', 'mean'),
        churn_rate      = ('Churn_Binary', 'mean')
    )
    .reset_index()
    .round(4))

rfm_segments.to_sql('rfm_segments', conn, if_exists='replace', index=False)
print(f"\n✅ Table 'rfm_segments' written — {len(rfm_segments)} rows")
print(rfm_segments.to_string(index=False))

conn.close()
print(f"\n✅ Database saved → {db_path}")

✅ Table 'customers' written — 7,043 rows
✅ Table 'churn_summary' written — 17 rows
      Contract tenure_group  churn_rate  avg_monthly  customer_count
Month-to-month         0-12      0.5135      58.2179            1994
Month-to-month        13-24      0.3772      69.3096             737
Month-to-month        25-36      0.3251      74.3262             486
Month-to-month        37-48      0.3354      78.4225             316
Month-to-month        49-72      0.2602      85.4458             342
      One year         0-12      0.1057      35.9285             123
      One year        13-24      0.0812      44.8787             197
      One year        25-36      0.0800      58.0988             250

✅ Table 'rfm_segments' written — 12 rows
 rfm_score  customer_count  avg_monthly  churn_rate
         3             518      22.3862      0.2973
         4             778      50.2903      0.4075
         5             936      65.7838      0.5000
         6             717      70.7708      0

## 10. Write SQL Query Functions to src/sql_queries.py
Three reusable query functions that any analyst or the Streamlit app can call.

In [11]:
sql_queries_code = '''"""
sql_queries.py — Reusable SQL query functions for the Churn Analysis SQLite DB.
Each function takes db_path as argument and returns a pandas DataFrame.

Interview talking point: Separating SQL logic into a module makes it testable,
reusable, and shows software engineering best practices alongside data science.
"""

import sqlite3
import pandas as pd


def get_high_value_churners(db_path: str) -> pd.DataFrame:
    """
    Returns customers who churned AND had MonthlyCharges > $70.
    These are the highest priority for win-back campaigns.
    """
    query = """
        SELECT
            customerID,
            MonthlyCharges,
            tenure,
            rfm_score,
            Churn
        FROM customers
        WHERE MonthlyCharges > 70
          AND Churn = 'Yes'
        ORDER BY MonthlyCharges DESC
    """
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)


def get_revenue_at_risk_by_segment(db_path: str) -> pd.DataFrame:
    """
    Total monthly revenue at risk per Contract type.
    Revenue at risk = sum of MonthlyCharges for churned customers.

    Interview talking point: This query directly answers the CFO question —
    'Where is our biggest revenue risk by segment?'
    """
    query = """
        SELECT
            CASE
                WHEN "Contract_Month-to-month" = 1 THEN 'Month-to-month'
                WHEN "Contract_One year"        = 1 THEN 'One year'
                WHEN "Contract_Two year"        = 1 THEN 'Two year'
                ELSE 'Unknown'
            END AS contract_type,
            COUNT(*)                    AS churned_customers,
            ROUND(SUM(MonthlyCharges), 2) AS total_monthly_revenue_lost,
            ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charge
        FROM customers
        WHERE Churn = 'Yes'
        GROUP BY contract_type
        ORDER BY total_monthly_revenue_lost DESC
    """
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)


def get_rfm_churn_summary(db_path: str) -> pd.DataFrame:
    """
    Churn rate and average charges per RFM score group.
    Shows whether higher RFM score customers churn less (validates the feature).
    """
    query = """
        SELECT
            rfm_score,
            COUNT(*)                          AS customer_count,
            ROUND(AVG(MonthlyCharges), 2)     AS avg_monthly_charges,
            ROUND(AVG(Churn_Binary) * 100, 1) AS churn_rate_pct
        FROM customers
        GROUP BY rfm_score
        ORDER BY rfm_score ASC
    """
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)
'''

with open('../src/sql_queries.py', 'w') as f:
    f.write(sql_queries_code)

print("✅ Saved → src/sql_queries.py")

✅ Saved → src/sql_queries.py


## 11. Test All SQL Queries
Run all three functions and verify they return correct results.

In [12]:
sys.path.insert(0, '../src')
from sql_queries import (
    get_high_value_churners,
    get_revenue_at_risk_by_segment,
    get_rfm_churn_summary
)

db_path = '../outputs/churn_analysis.db'

# ── Query 1 ─────────────────────────────────────────────────────────────
print("=" * 55)
print("QUERY 1 — High Value Churners (MonthlyCharges > $70)")
print("=" * 55)
q1 = get_high_value_churners(db_path)
print(f"Rows returned: {len(q1)}")
print(q1.head(10).to_string(index=False))

# ── Query 2 ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("QUERY 2 — Revenue at Risk by Contract Type")
print("=" * 55)
q2 = get_revenue_at_risk_by_segment(db_path)
print(q2.to_string(index=False))

# ── Query 3 ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("QUERY 3 — RFM Score vs Churn Rate")
print("=" * 55)
q3 = get_rfm_churn_summary(db_path)
print(q3.to_string(index=False))
print("\n✅ All queries executed successfully.")

QUERY 1 — High Value Churners (MonthlyCharges > $70)
Rows returned: 1267
customerID  MonthlyCharges  tenure  rfm_score Churn
8199-ZLLSA          118.35      67         12   Yes
2889-FPWRM          117.80      72         12   Yes
2302-ANTDP          117.45      48          9   Yes
9053-JZFKV          116.20      67         14   Yes
1444-VVSGW          115.65      70         12   Yes
0201-OAMXR          115.55      70         12   Yes
4361-BKAXE          114.50      41          9   Yes
1555-DJEQW          114.20      70         14   Yes
9158-VCTQB          113.60      41          9   Yes
7279-BUYWN          113.20      41          9   Yes

QUERY 2 — Revenue at Risk by Contract Type
 contract_type  churned_customers  total_monthly_revenue_lost  avg_monthly_charge
Month-to-month               1655                   120847.10               73.02
      One year                166                    14118.45               85.05
      Two year                 48                     4165.30    

## 12. Stage 2 Complete — Summary

In [13]:
print("""
╔══════════════════════════════════════════════════════════╗
║           STAGE 2 COMPLETE — OUTPUTS CREATED            ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  data/telco_features.csv                                 ║
║    → Full feature set ready for modeling                 ║
║    → Includes RFM scores, engineered flags, OHE cols     ║
║                                                          ║
║  outputs/churn_analysis.db (SQLite)                      ║
║    → Table: customers     (all features)                 ║
║    → Table: churn_summary (by contract + tenure)         ║
║    → Table: rfm_segments  (by RFM score)                 ║
║                                                          ║
║  src/sql_queries.py                                      ║
║    → get_high_value_churners()                           ║
║    → get_revenue_at_risk_by_segment()                    ║
║    → get_rfm_churn_summary()                             ║
║                                                          ║
╠══════════════════════════════════════════════════════════╣
║  Next: Stage 3 — XGBoost Modeling                        ║
╚══════════════════════════════════════════════════════════╝
""")

# Final checks
import os
checks = {
    'data/telco_features.csv'    : '../data/telco_features.csv',
    'outputs/churn_analysis.db'  : '../outputs/churn_analysis.db',
    'src/sql_queries.py'         : '../src/sql_queries.py',
}
for label, path in checks.items():
    status = "✅" if os.path.exists(path) else "❌ MISSING"
    print(f"  {status}  {label}")


╔══════════════════════════════════════════════════════════╗
║           STAGE 2 COMPLETE — OUTPUTS CREATED            ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  data/telco_features.csv                                 ║
║    → Full feature set ready for modeling                 ║
║    → Includes RFM scores, engineered flags, OHE cols     ║
║                                                          ║
║  outputs/churn_analysis.db (SQLite)                      ║
║    → Table: customers     (all features)                 ║
║    → Table: churn_summary (by contract + tenure)         ║
║    → Table: rfm_segments  (by RFM score)                 ║
║                                                          ║
║  src/sql_queries.py                                      ║
║    → get_high_value_churners()                           ║
║    → get_revenue_at_risk_by_segment()                    ║
║    → get_rfm_churn_sum